## llama.cpp GGUF backend — Kaggle T4 x2

Run cells in order.

In [ ]:

# Notebook bootstrap panel: compact realtime output for clone/install before gguf_backend is available.
import os, sys, time, html, uuid, subprocess, select
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Javascript

BOOT_TERM_ID = "boot_term_" + uuid.uuid4().hex
_boot_title = widgets.HTML('<div style="font-family:ui-monospace,monospace;font-weight:700;padding:8px 10px;border:1px solid #333;border-bottom:0;background:#111;color:#f5f5f5;">Bootstrap</div>')
_boot_status = widgets.HTML('<div style="font-family:ui-monospace,monospace;padding:8px 10px;border-left:1px solid #333;border-right:1px solid #333;background:#181818;color:#ddd;">idle</div>')
_boot_term = widgets.HTML(f'<div id="{BOOT_TERM_ID}" style="margin:0;padding:10px;height:260px;overflow-y:auto;white-space:pre-wrap;word-break:break-word;background:#050505;color:#e8e8e8;border:1px solid #333;font-family:ui-monospace,monospace;font-size:13px;line-height:1.35;">ready</div>')
_boot_footer = widgets.HTML('<div style="font-family:ui-monospace,monospace;font-size:12px;padding:7px 10px;border:1px solid #333;border-top:0;background:#111;color:#aaa;">log: -</div>')
display(widgets.VBox([_boot_title, _boot_status, _boot_term, _boot_footer]))
display(Javascript(f"""
(function() {{
  const id = "{BOOT_TERM_ID}";
  const key = "__autoscr_" + id;
  if (window[key]) clearInterval(window[key]);
  window[key] = setInterval(function() {{
    const el = document.getElementById(id);
    if (el) el.scrollTop = el.scrollHeight;
  }}, 100);
}})();
"""))

def _esc(x):
    return html.escape(str(x), quote=False)

def _set_boot_status(text):
    _boot_status.value = f'<div style="font-family:ui-monospace,monospace;padding:8px 10px;border-left:1px solid #333;border-right:1px solid #333;background:#181818;color:#ddd;">{_esc(text)}</div>'

def _set_boot_term(lines):
    body = "\n".join(lines[-300:])
    _boot_term.value = f'<div id="{BOOT_TERM_ID}" style="margin:0;padding:10px;height:260px;overflow-y:auto;white-space:pre-wrap;word-break:break-word;background:#050505;color:#e8e8e8;border:1px solid #333;font-family:ui-monospace,monospace;font-size:13px;line-height:1.35;">{_esc(body)}</div>'

def bootstrap_run(cmd, *, label="bootstrap", cwd=None, check=True):
    log_dir = Path("/kaggle/working/_logs" if Path("/kaggle/working").exists() else "/content/_logs")
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (label.replace(" ", "_") + ".log")
    _boot_footer.value = f'<div style="font-family:ui-monospace,monospace;font-size:12px;padding:7px 10px;border:1px solid #333;border-top:0;background:#111;color:#aaa;">log: {_esc(log_path)}</div>'
    lines = ["$ " + (cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))]
    _set_boot_term(lines)
    _set_boot_status(f"running: {label}")
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, bufsize=0)
    fd = p.stdout.fileno()
    os.set_blocking(fd, False)
    start = time.time(); last = 0
    with open(log_path, "w", encoding="utf-8", errors="replace") as f:
        while True:
            ready, _, _ = select.select([fd], [], [], 0.1)
            if ready:
                raw = os.read(fd, 8192)
                if raw:
                    text = raw.decode("utf-8", errors="replace")
                    f.write(text); f.flush()
                    for line in text.replace("\r", "\n").splitlines():
                        line = line.strip()
                        if line:
                            if len(line) > 280: line = line[:280] + " ..."
                            lines.append(line)
            if time.time() - last > 0.15:
                _set_boot_status(f"running: {label} | elapsed {int(time.time()-start)}s")
                _set_boot_term(lines)
                last = time.time()
            if p.poll() is not None:
                break
    rc = p.wait()
    lines.append(f"$ exit {rc}")
    _set_boot_term(lines)
    _set_boot_status(f"completed: {label}" if rc == 0 else f"failed: {label} | exit {rc}")
    if check and rc != 0:
        raise RuntimeError(f"{label} failed, check log: {log_path}")
    return rc

# CELL 1 — setup repo
REPO_URL = "https://github.com/N3iKos/llama-cpp-notebook"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/llama-cpp-notebook")

if not REPO_DIR.exists():
    bootstrap_run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], label="clone repo")
else:
    bootstrap_run(["git", "-C", str(REPO_DIR), "pull"], label="update repo", check=False)

bootstrap_run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], label="install package")
sys.path.insert(0, str(REPO_DIR))

from gguf_backend.panel import show_summary
show_summary("repo ready", lines=[f"repo: {REPO_DIR}", "module: gguf_backend"])


In [ ]:
# CELL 2 — diagnostics
from gguf_backend.shell import run

run("nvidia-smi --query-gpu=index,name,memory.total,memory.free,driver_version,compute_cap --format=csv,noheader,nounits || true")
run("nvidia-smi topo -m || true")
run("nvcc --version || true")
run("df -h /kaggle/working /tmp || true")

In [ ]:

# CELL 3 — install prebuilt llama.cpp
from gguf_backend.installer import ensure_apt_tools, install_llama_cpp_prebuilt
from gguf_backend.shell import run
from gguf_backend.panel import show_summary

ROOT = "/kaggle/working"

ensure_apt_tools()
info = install_llama_cpp_prebuilt(ROOT, cuda_preference="12.8", force=False)

show_summary("llama.cpp install info", info)
run(f'"{info["server"]}" --version', label="llama-server version")
run(f'"{info["server"]}" --list-devices', label="llama-server devices")


In [ ]:

# CELL 4 — download model and optional mmproj
import os
from gguf_backend.downloader import download_model_pair
from gguf_backend.panel import show_summary

MODEL_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf"
MMPROJ_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/mmproj-Qwen2.5-VL-3B-Instruct-Q8_0.gguf"

MODEL_DIR = "/kaggle/working/models/current"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

cfg = download_model_pair(MODEL_URL, MMPROJ_URL, MODEL_DIR, hf_token=HF_TOKEN, connections=16)
show_summary("download config", cfg)


In [ ]:
# CELL 5 — config, start server, warmup
import json
from pathlib import Path
from gguf_backend.server import ServerConfig, start_server

ROOT = "/kaggle/working"
cfg = json.loads(Path("/kaggle/working/model_config.json").read_text())

CTX_SIZE = 8192
SPLIT_MODE = "row"          # row, layer, none
FALLBACK_SPLIT_MODE = "layer"
TENSOR_SPLIT = "1,1"

BATCH_SIZE = 2048
UBATCH_SIZE = 512
PARALLEL = 1
FLASH_ATTN = True
CACHE_TYPE_K = "f16"
CACHE_TYPE_V = "f16"

IMAGE_MIN_TOKENS = None
IMAGE_MAX_TOKENS = None
CHAT_TEMPLATE_KWARGS = None  # example: '{"enable_thinking":true}'

server_cfg = ServerConfig(
    root_dir=ROOT,
    model_path=cfg["model_path"],
    mmproj_path=cfg.get("mmproj_path", ""),
    port=8080,
    alias="local-vl",
    ctx_size=CTX_SIZE,
    split_mode=SPLIT_MODE,
    fallback_split_mode=FALLBACK_SPLIT_MODE,
    tensor_split=TENSOR_SPLIT,
    batch_size=BATCH_SIZE,
    ubatch_size=UBATCH_SIZE,
    parallel=PARALLEL,
    flash_attn=FLASH_ATTN,
    cache_type_k=CACHE_TYPE_K,
    cache_type_v=CACHE_TYPE_V,
    image_min_tokens=IMAGE_MIN_TOKENS,
    image_max_tokens=IMAGE_MAX_TOKENS,
    chat_template_kwargs=CHAT_TEMPLATE_KWARGS,
    cuda_visible_devices="0,1",
)

server_info = start_server(server_cfg, warmup=True)
from gguf_backend.panel import show_summary
show_summary("server info", server_info)


In [ ]:

# CELL 6 — text response test
import json
from gguf_backend.client import chat
from gguf_backend.panel import show_summary

status, resp = chat(
    "http://127.0.0.1:8080",
    "local-vl",
    "Tulis satu kalimat bahwa backend siap digunakan.",
    max_tokens=80,
)

show_summary("text response test", lines=[f"status: {status}", json.dumps(resp, indent=2, ensure_ascii=False)[:3000]])


In [ ]:
# CELL 7 — tunnels
from gguf_backend.tunnel import start_tunnels

TUNNEL_MODE = "both"  # both, ngrok, cloudflare
NGROK_AUTHTOKEN = ""

if not NGROK_AUTHTOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        NGROK_AUTHTOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    except Exception:
        NGROK_AUTHTOKEN = ""

urls = start_tunnels(
    8080,
    "/kaggle/working",
    mode=TUNNEL_MODE,
    ngrok_token=NGROK_AUTHTOKEN,
    fallback_cloudflare=True,
)

from gguf_backend.panel import show_summary
show_summary("tunnel result", urls)


In [ ]:
# CELL 8 — stop/status utilities
from gguf_backend.server import stop_server
from gguf_backend.shell import run

# stop_server("/kaggle/working")
run("nvidia-smi --query-gpu=index,name,memory.used,memory.free,utilization.gpu,power.draw --format=csv,noheader,nounits || true")
run("cat /kaggle/working/llama_server.pid 2>/dev/null || true")